In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
SOURCE_TABLE = f"{CATALOG}.bronze.eia_natural_gas_prices"
TARGET_TABLE = f"{CATALOG}.silver.eia_gas_prices_silver"
WINDOW_COL   = "period"
OVERLAP_DAYS = 7          
KEYS = ["series", "period"]

dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])
mode = dbutils.widgets.get("mode")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  series             STRING,
  series_description STRING,
  product            STRING,
  product_name       STRING,
  process            STRING,
  duoarea            STRING,
  area_name          STRING,
  period             DATE,
  price_usd_mmbtu    DOUBLE,
  units              STRING,
  frequency          STRING,
  silver_updated_at  TIMESTAMP
)
USING DELTA
PARTITIONED BY (period)
""")

def merge_into(table, df, keys):
    df.createOrReplaceTempView("stg")
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    spark.sql(f"""
      MERGE INTO {table} t
      USING stg s
      ON {on}
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

# --- how much bronze to read? --------------------------------------------------
# first run (silver empty) -> ALL of bronze, so history is never missed
# later runs               -> from the last period already loaded, with overlap
# mode=full                -> ignore watermark, re-read everything (repair)
last_loaded = spark.table(TARGET_TABLE).agg(F.max(WINDOW_COL)).first()[0]

if mode == "full" or last_loaded is None:
    src = spark.table(SOURCE_TABLE)
    print("[silver:eia] reading ALL bronze rows (first run or mode=full)")
else:
    src = (spark.table(SOURCE_TABLE)
           .where(F.col(WINDOW_COL) >= F.date_sub(F.lit(last_loaded), OVERLAP_DAYS)))
    print(f"[silver:eia] incremental: bronze from {last_loaded} (-{OVERLAP_DAYS}d overlap)")

# --- keep only the columns we want -------------------------------------------
out = (src
    .select(
        F.col("series"),
        F.col("series_description"),
        F.col("product"),
        F.col("product_name"),
        F.col("process"),
        F.col("duoarea"),
        F.col("area_name"),
        F.to_date("period").alias("period"),
        F.col("price_usd_mmbtu").cast("double"),
        F.col("units"),
        F.col("frequency"),
        F.current_timestamp().alias("silver_updated_at"),
    )
    .dropDuplicates(KEYS)
    .cache())

# --- data quality: fail loudly BEFORE writing ---------------------------------
n = out.count()
assert n > 0, "[silver:eia] no rows read from bronze - check bronze table"

for k in KEYS:
    bad = out.where(F.col(k).isNull()).count()
    assert bad == 0, f"[silver:eia] {bad} rows with null in key {k}"

# EIA gaps are normal: null prices are reported, not failed
null_prices = out.where(F.col("price_usd_mmbtu").isNull()).count()
print(f"INFO [silver:eia] {null_prices} null-price rows (EIA gaps, kept)")

bad_price = out.where(
    F.col("price_usd_mmbtu").isNotNull()
    & ((F.col("price_usd_mmbtu") < 0.0001) | (F.col("price_usd_mmbtu") > 1000000))
).count()
assert bad_price == 0, f"[silver:eia] {bad_price} rows of price_usd_mmbtu outside [0.0001, 1000000]"

# EIA publishes this report WEEKLY (Wed/Thu), not daily -> allow two report cycles
fresh = out.where(F.col("period") >= F.date_sub(F.current_date(), 14)).count()
assert fresh > 0, "[silver:eia] freshness fail: no period within 14 days - weekly EIA feed looks dead"

# --- write --------------------------------------------------------------------
merge_into(TARGET_TABLE, out, KEYS)
latest = spark.table(TARGET_TABLE).agg(F.max("period")).first()[0]
print(f"[silver:eia] processed={n}, total={spark.table(TARGET_TABLE).count()}, latest period={latest}")
out.unpersist()